In [10]:
"""
YouTube Video Summarizer
------------------------
Fetches a YouTube video's transcript and summarizes it using the OpenAI API.

Setup:
    %pip install -q youtube-transcript-api openai

    Set your OpenAI API key as an environment variable:
        export OPENAI_API_KEY="your-key-here"

Usage:
    python youtube_summarizer.py <youtube_url_or_video_id>

Example:
    python youtube_summarizer.py https://www.youtube.com/watch?v=dQw4w9WgXcQ
    python youtube_summarizer.py dQw4w9WgXcQ
"""

import argparse
import os
import re
import sys

%pip install -q youtube-transcript-api openai

from youtube_transcript_api import (
    YouTubeTranscriptApi,
    TranscriptsDisabled,
    NoTranscriptFound,
    VideoUnavailable,
)
from openai import OpenAI


def extract_video_id(url_or_id: str) -> str:
    """Pull the 11-character video ID out of a full YouTube URL, or return the input as-is."""
    patterns = [
        r"(?:v=|\/)([0-9A-Za-z_-]{11}).*",  # watch?v= or youtu.be/
        r"^([0-9A-Za-z_-]{11})$",           # bare video ID
    ]
    for pattern in patterns:
        match = re.search(pattern, url_or_id)
        if match:
            return match.group(1)
    raise ValueError(f"Could not extract a video ID from: {url_or_id}")


def fetch_transcript(video_id: str) -> str:
    """Fetch and flatten the transcript into a single block of text."""
    try:
        transcript = YouTubeTranscriptApi().fetch(video_id, languages=["en"])
    except TranscriptsDisabled:
        raise RuntimeError("This video has transcripts disabled.")
    except NoTranscriptFound:
        raise RuntimeError("No transcript could be found for this video.")
    except VideoUnavailable:
        raise RuntimeError("This video is unavailable (private, deleted, or region-locked).")

    if hasattr(transcript, "to_raw_data"):
        transcript_data = transcript.to_raw_data()
    else:
        transcript_data = transcript

    return " ".join(chunk["text"] for chunk in transcript_data)


def summarize_transcript(transcript: str, model: str = "gpt-4.1-mini") -> str:
    """Send the transcript to OpenAI and return a summary."""
    api_key = os.getenv("OPENAI_API_KEY")
    if not api_key:
        raise RuntimeError("Set OPENAI_API_KEY to use the summarizer.")

    client = OpenAI(api_key=api_key)

    prompt = (
        "Summarize the following YouTube video transcript. Structure your response as:\n"
        "1. A 2-3 sentence overview\n"
        "2. Key points (bulleted)\n"
        "3. Any notable quotes, conclusions, or calls to action\n\n"
        f"Transcript:\n{transcript}"
    )

    response = client.chat.completions.create(
        model=model,
        max_tokens=1000,
        messages=[{"role": "user", "content": prompt}],
    )

    return response.choices[0].message.content or ""


def main():
    parser = argparse.ArgumentParser(description="Summarize a YouTube video using its transcript and OpenAI.")
    parser.add_argument("video", help="YouTube URL or video ID")
    parser.add_argument("--model", default="gpt-4.1-mini", help="OpenAI model to use (default: gpt-4.1-mini)")
    args = parser.parse_args()

    try:
        video_id = extract_video_id(args.video)
        print(f"Fetching transcript for video ID: {video_id}...", file=sys.stderr)
        transcript = fetch_transcript(video_id)

        print("Summarizing with OpenAI...", file=sys.stderr)
        summary = summarize_transcript(transcript, model=args.model)

        print("\n" + "=" * 60)
        print("SUMMARY")
        print("=" * 60)
        print(summary)

    except (ValueError, RuntimeError) as e:
        print(f"Error: {e}", file=sys.stderr)
        sys.exit(1)


if __name__ == "__main__":
    if len(sys.argv) > 1 and not any(arg.startswith("-") for arg in sys.argv[1:]):
        main()


Note: you may need to restart the kernel to use updated packages.


Fetching transcript for video ID: cAI9rr_0-mU...
Summarizing with OpenAI...



SUMMARY
1. **Overview:**  
This video is a detailed live trading stream and market analysis focused on the effects of recent CPI data and the Federal Reserve Chairman Kevin Walsh’s testimony to Congress. The host discusses trading strategies using Edgeville AI reports, explains market behavior in volatile conditions, shares experiences with prop trading firms, and emphasizes caution during high-impact news events.

2. **Key Points:**  
- The morning’s CPI data caused significant market volatility; however, the host does not rely heavily on CPI data for trading decisions, viewing market moves as influenced more by market makers and algorithms than raw numbers.  
- Kevin Walsh's testimony to Congress, spanning two hours today and the same again tomorrow, is regarded as a major market event causing sharp and unpredictable moves driven by headline reactions and algorithmic trading.  
- Trading strategies have been adjusted for increased volatility: entries are now taken only at the 50% le

In [9]:
import sys

sys.argv = ["youtube_summarizer.py", "https://www.youtube.com/watch?v=cAI9rr_0-mU"]
main()


Fetching transcript for video ID: cAI9rr_0-mU...
Summarizing with Claude...
Error: Set ANTHROPIC_API_KEY to use the summarizer.


SystemExit: 1